# MgO Phonon Workflow — `ph_runner` Demo

A complete end-to-end phonon workflow for MgO (rock-salt, ibrav = 2 FCC):

| Step | Code | What we get |
|------|------|-------------|
| 1 | `pw.x` | SCF ground state |
| 2 | `ph.x` at Γ | Γ-point phonons + Born charges (ε∞, Z*) |
| 3 | `dynmat.x` | TO modes — no q direction (ASR only) |
| 4 | `dynmat.x` | LO/TO modes — q → [1, 0, 0] (non-analytic correction) |
| 5 | `ph.x` 2×2×2 | Full phonon grid → dynamical matrices |
| 6 | `q2r.x` | Fourier-transform to interatomic force constants |
| 7 | `matdyn.x` | Phonon dispersion along Γ→X→W→K→Γ→L |
| 8 | `matdyn.x` | Phonon DOS on a 16×16×16 q-mesh |

All runners from `ph_runner.py`; input builders from `pw_input.py` and `ph_input.py`.

In [ ]:
from pathlib import Path
import glob, shutil, os

os.environ['OMP_NUM_THREADS'] = '1'

RUN_ROOT   = Path('.').resolve()
PSEUDO_DIR = RUN_ROOT / 'pseudo'

# QE executables — find via build directory, fall back to PATH
_pw_candidates = sorted(glob.glob('/home/*/repositories/q-e/*/bin/pw.x'))
_qe_bin = Path(_pw_candidates[0]).parent if _pw_candidates else None

def _find_exe(name):
    if _qe_bin:
        c = _qe_bin / name
        if c.is_file():
            return [str(c)]
    p = shutil.which(name)
    if p:
        return [p]
    raise RuntimeError(f'{name} not found in build dir or PATH.')

PW_CMD     = _find_exe('pw.x')
PH_CMD     = _find_exe('ph.x')
DYNMAT_CMD = _find_exe('dynmat.x')
Q2R_CMD    = _find_exe('q2r.x')
MATDYN_CMD = _find_exe('matdyn.x')

RUN_DIR = RUN_ROOT / 'mgo_phonon_demo'
OUT_DIR = RUN_DIR / 'out'
PH_DIR  = RUN_DIR / 'ph'
for d in [RUN_DIR, OUT_DIR, PH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'pw.x     : {PW_CMD[0]}')
print(f'ph.x     : {PH_CMD[0]}')
print(f'dynmat.x : {DYNMAT_CMD[0]}')
print(f'q2r.x    : {Q2R_CMD[0]}')
print(f'matdyn.x : {MATDYN_CMD[0]}')
print(f'RUN_DIR  : {RUN_DIR}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.build import bulk

from pw_input import (
    ControlNamelist, SystemNamelist, ElectronsNamelist,
    AtomicSpeciesCard, AtomicPositionsCard, KPointsAutoCard, PWInput,
)
from ph_input import PhInputph, DynmatInput, Q2rInput, MatdynInput, QPointPath
from convergence_runner import QERunner
from ph_runner import PhRunner, DynmatRunner, Q2rRunner, MatdynRunner

---
## 1. MgO SCF with `pw.x`

MgO rock-salt: FCC Bravais lattice (ibrav = 2), 2-atom basis.  
PAW pseudopotentials require higher cutoffs: 80 Ry / 640 Ry.

In [ ]:
mgo = bulk('MgO', 'rocksalt', a=4.211)
print(f'MgO: a = {mgo.cell.lengths()[0]:.4f} Å,  nat = {len(mgo)}')
print('Crystal positions:')
for sym, pos in zip(mgo.get_chemical_symbols(), mgo.get_scaled_positions()):
    print(f'  {sym}  {pos}')

PSEUDOS = {
    'Mg': 'Mg.pbe-spnl-kjpaw_psl.1.0.0.UPF',
    'O':  'O.pbe-n-kjpaw_psl.1.0.0.UPF',
}

mgo_scf = PWInput(
    control=ControlNamelist(
        calculation='scf',
        prefix='mgo',
        pseudo_dir=str(PSEUDO_DIR),
        outdir=str(OUT_DIR),
    ),
    system=SystemNamelist.from_atoms(
        mgo, ibrav=2,
        ecutwfc=80.0, ecutrho=640.0,
    ),
    electrons=ElectronsNamelist(conv_thr=1e-10),
    atomic_species=AtomicSpeciesCard.from_atoms(mgo, PSEUDOS),
    atomic_positions=AtomicPositionsCard.from_atoms(mgo, units='crystal'),
    k_points=KPointsAutoCard(2, nk=8),
)
print()
print(mgo_scf.to_string())

In [ ]:
pw    = QERunner(PW_CMD)
r_scf = pw.run_one('mgo_scf', mgo_scf, RUN_DIR)

wall = f"{r_scf['wall_s']:.1f} s" if not np.isnan(r_scf['wall_s']) else '(cached)'
print(f"Total energy : {r_scf['energy_ry']:.6f} Ry")
print(f"k-points irr : {r_scf['nk_irr']}")
print(f"Wall time    : {wall}")

---
## 2. Phonons at Γ with `ph.x`

`with_dielectric()` adds `epsil=.true., zeu=.true.` to compute ε∞ and Born charges Z*,
both stored in the dynamical matrix file and required by `dynmat.x` for LO-TO splitting.

In [ ]:
FILDYN_G = os.path.relpath(PH_DIR / 'mgo.dynG')

ph_inp = (
    PhInputph.single_q(
        prefix='mgo',
        qpoint=(0, 0, 0),
        fildyn=FILDYN_G,
        outdir=str(OUT_DIR),
        tr2_ph=1e-14,
    )
    .with_dielectric()
)
print(ph_inp.to_string())

In [ ]:
ph   = PhRunner(PH_CMD)
r_ph = ph.run_one('mgo_gamma', ph_inp, PH_DIR)

wall = f"{r_ph['wall_s']:.1f} s" if not np.isnan(r_ph['wall_s']) else '(cached)'
print(f'Wall time : {wall}')
if r_ph['frequencies_cm'] is not None:
    print('Frequencies at Γ (cm⁻¹):')
    for i, f in enumerate(r_ph['frequencies_cm'], 1):
        print(f'  mode {i:2d}  {f:10.4f}')

---
## 3. `dynmat.x` — TO modes (no q direction)

Without a q-direction the non-analytic Coulomb correction is not applied:
all 3 optical modes stay **degenerate at the TO frequency**.
`lperm=True` additionally computes the dielectric permittivity from Born charges.

In [ ]:
dm_to = DynmatInput(fildyn=FILDYN_G, asr='crystal', lperm=True)
print(dm_to.to_string())

In [ ]:
dyn  = DynmatRunner(DYNMAT_CMD)
r_to = dyn.run_one('dynmat_to', dm_to, PH_DIR)

wall = f"{r_to['wall_s']:.1f} s" if not np.isnan(r_to['wall_s']) else '(cached)'
print(f'Wall time : {wall}\n')

print(f"{'Mode':>5}  {'freq (cm⁻¹)':>13}  {'freq (THz)':>11}  {'IR':>10}")
print('-' * 47)
for i, (fc, ft, ir) in enumerate(
    zip(r_to['freq_cm'], r_to['freq_thz'], r_to['ir']), 1
):
    print(f'  {i:3d}   {fc:12.4f}   {ft:10.4f}   {ir:10.4f}')

if r_to['eps_electronic'] is not None:
    print('\nε∞ (electronic):')
    for row in r_to['eps_electronic']:
        print('  ', '  '.join(f'{x:8.4f}' for x in row))

if r_to['eps_static'] is not None:
    print('\nε₀ (static = ε∞ + ionic):')
    for row in r_to['eps_static']:
        print('  ', '  '.join(f'{x:8.4f}' for x in row))

---
## 4. `dynmat.x` — LO/TO split along q → [1, 0, 0]

The non-analytic correction lifts the 3-fold optical degeneracy:
- **2 TO modes** (transverse, perpendicular to q) — unchanged
- **1 LO mode** (longitudinal, along q) — shifted up by ~300 cm⁻¹

The large LO-TO splitting in MgO reflects its strong ionicity.

In [ ]:
dm_lo = DynmatInput(fildyn=FILDYN_G, asr='crystal', lperm=True, q=(1, 0, 0))
print(dm_lo.to_string())

In [ ]:
r_lo = dyn.run_one('dynmat_lo100', dm_lo, PH_DIR)

wall = f"{r_lo['wall_s']:.1f} s" if not np.isnan(r_lo['wall_s']) else '(cached)'
print(f'Wall time : {wall}\n')

print(f"{'Mode':>5}  {'freq (cm⁻¹)':>13}  {'freq (THz)':>11}  {'IR':>10}")
print('-' * 47)
for i, (fc, ft, ir) in enumerate(
    zip(r_lo['freq_cm'], r_lo['freq_thz'], r_lo['ir']), 1
):
    print(f'  {i:3d}   {fc:12.4f}   {ft:10.4f}   {ir:10.4f}')

if r_lo['eps_electronic'] is not None:
    print('\nε∞ (electronic):')
    for row in r_lo['eps_electronic']:
        print('  ', '  '.join(f'{x:8.4f}' for x in row))

if r_lo['eps_static'] is not None:
    print('\nε₀ along [100]:')
    for row in r_lo['eps_static']:
        print('  ', '  '.join(f'{x:8.4f}' for x in row))

if r_lo['eps_ionic'] is not None:
    print('\nΔε (ionic = ε₀ − ε∞):')
    for row in r_lo['eps_ionic']:
        print('  ', '  '.join(f'{x:8.4f}' for x in row))

---
## 5. LO-TO splitting summary

In [ ]:
freq_to = r_to['freq_cm']
freq_lo = r_lo['freq_cm']

print(f"{'Mode':>5}  {'no q (cm⁻¹)':>14}  {'q=[100] (cm⁻¹)':>16}  {'Δ (cm⁻¹)':>11}")
print('-' * 55)
for i, (f1, f2) in enumerate(zip(freq_to, freq_lo), 1):
    delta = f2 - f1
    mark  = '  ← LO' if abs(delta) > 50 else ''
    print(f'  {i:3d}   {f1:12.4f}   {f2:14.4f}   {delta:10.4f}{mark}')

opt_lo  = freq_lo[freq_lo > 10]
lo_freq = opt_lo.max()
to_freq = opt_lo[opt_lo < lo_freq].mean()
print(f'\nTO frequency : {to_freq:.2f} cm⁻¹')
print(f'LO frequency : {lo_freq:.2f} cm⁻¹')
print(f'LO-TO split  : {lo_freq - to_freq:.2f} cm⁻¹')

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))

opt_to_vals = freq_to[freq_to > 10]
for f in np.unique(np.round(opt_to_vals, 0)):
    lbl = 'TO' if f == np.unique(np.round(opt_to_vals, 0))[0] else ''
    ax.hlines(f, 0.2, 0.45, lw=2.5, color='steelblue', label=lbl)

for f in np.unique(np.round(opt_lo, 0)):
    color = 'tomato' if f > to_freq + 50 else 'steelblue'
    label = 'LO' if f > to_freq + 50 else ''
    ax.hlines(f, 0.55, 0.8, lw=2.5, color=color, label=label)

ax.set_xlim(0, 1)
ax.set_xticks([0.325, 0.675])
ax.set_xticklabels(['no q direction', 'q → [100]'])
ax.set_ylabel('Frequency (cm⁻¹)')
ax.set_title('MgO optical modes at Γ')
ax.legend(loc='center right')
plt.tight_layout()
plt.show()

---
## 6. `ph.x` dispersion on a 2×2×2 q-grid

With `ldisp=.true.` ph.x computes phonons on a uniform Monkhorst-Pack q-grid and
writes one dynamical matrix file per q-point: `mgo.dyn1`, `mgo.dyn2`, …  
A 2×2×2 grid is the minimum for a non-trivial dispersion; 4×4×4 or denser gives
research-quality IFC but takes proportionally longer.

`with_dielectric()` is again needed so that the LO-TO non-analytic correction
is available to matdyn.x when interpolating near Γ.

In [ ]:
FILDYN_DISP = os.path.relpath(PH_DIR / 'mgo.dyn')

ph_disp_inp = (
    PhInputph.dispersion(
        prefix='mgo',
        nq1=2, nq2=2, nq3=2,
        fildyn=FILDYN_DISP,
        outdir=str(OUT_DIR),
        tr2_ph=1e-14,
    )
    .with_dielectric()
)
print(ph_disp_inp.to_string())

In [ ]:
r_ph_disp = ph.run_one('mgo_disp', ph_disp_inp, PH_DIR)

wall = f"{r_ph_disp['wall_s']:.1f} s" if not np.isnan(r_ph_disp['wall_s']) else '(cached)'
print(f'Wall time : {wall}\n')

dyn_files = sorted(PH_DIR.glob('mgo.dyn[0-9]*'))
print(f'Dynamical matrix files written ({len(dyn_files)}):')
for f in dyn_files:
    print(f'  {f.name}')

---
## 7. `q2r.x` — Fourier transform to interatomic force constants

`q2r.x` reads all the `mgo.dynN` files and Fourier-transforms them to real-space
interatomic force constants stored in `mgo.fc`.  
`zasr='crystal'` enforces the acoustic sum rule in a translationally-invariant way.

In [ ]:
FLFRC = os.path.relpath(PH_DIR / 'mgo.fc')

q2r_inp = Q2rInput(
    fildyn=FILDYN_DISP,
    flfrc=FLFRC,
    zasr='crystal',
)
print(q2r_inp.to_string())

In [ ]:
q2r   = Q2rRunner(Q2R_CMD)
r_q2r = q2r.run_one('mgo_q2r', q2r_inp, PH_DIR)

wall = f"{r_q2r['wall_s']:.1f} s" if not np.isnan(r_q2r['wall_s']) else '(cached)'
print(f'q2r.x done : {wall}')
fc_path = Path(FLFRC)
if fc_path.exists():
    print(f'Force constants file : {fc_path.name}  ({fc_path.stat().st_size / 1024:.0f} kB)')

---
## 8. `matdyn.x` — phonon dispersion along Γ→X→W→K→Γ→L

matdyn.x interpolates the IFC from `mgo.fc` onto any q-path using a Fourier back-transform.
The FCC BZ path (crystal coordinates, `q_in_cryst_coord=.true.`):

| Label | Crystal coords | Character |
|-------|---------------|----------|
| Γ | (0, 0, 0) | zone centre |
| X | (½, 0, ½) | zone boundary |
| W | (½, ¼, ¾) | zone edge |
| K | (⅜, ⅜, ¾) | zone edge |
| Γ | (0, 0, 0) | back to centre |
| L | (0, ½, ½) | zone boundary |

In [ ]:
FLFRQ = os.path.relpath(PH_DIR / 'mgo.freq')
FLVEC = os.path.relpath(PH_DIR / 'mgo.modes')

fcc_path = (
    QPointPath(cryst=True, band_form=True)
    .add(0.000, 0.000, 0.000, name='Γ', npoints=40)
    .add(0.500, 0.000, 0.500, name='X', npoints=40)
    .add(0.500, 0.250, 0.750, name='W', npoints=40)
    .add(0.375, 0.375, 0.750, name='K', npoints=40)
    .add(0.000, 0.000, 0.000, name='Γ', npoints=40)
    .add(0.000, 0.500, 0.500, name='L', npoints=1)
)

md_disp_inp = MatdynInput.dispersion(
    flfrc=FLFRC,
    qpath=fcc_path,
    asr='crystal',
    flfrq=FLFRQ,
    flvec=FLVEC,
)
print(md_disp_inp.to_string())

In [ ]:
matdyn   = MatdynRunner(MATDYN_CMD)
r_disp   = matdyn.run_one('mgo_matdyn_disp', md_disp_inp, PH_DIR)
res_disp = r_disp['results']

wall = f"{r_disp['wall_s']:.1f} s" if not np.isnan(r_disp['wall_s']) else '(cached)'
print(f'Wall time : {wall}')
print(res_disp)

In [ ]:
path      = res_disp.path
seg_pts   = [40, 40, 40, 40, 40, 1]
hs_idx    = [0] + list(np.cumsum(seg_pts[:-1]))
hs_labels = ['Γ', 'X', 'W', 'K', 'Γ', 'L']
xticks    = [(path[i], lbl) for i, lbl in zip(hs_idx, hs_labels)]

fig, ax = plt.subplots(figsize=(6, 5))
res_disp.plot_dispersion(ax=ax, xticks=xticks)
ax.set_title('MgO phonon dispersion  (2×2×2 q-grid IFC)')
plt.tight_layout()
plt.show()

---
## 9. `matdyn.x` — phonon DOS on a 16×16×16 q-mesh

matdyn.x can compute the DOS on any q-mesh independently of the ph.x grid
because it only needs the IFC file.  A 16×16×16 mesh is dense enough for
a well-converged DOS even from a 2×2×2 IFC.  
`degauss=5.0` applies Gaussian broadening (cm⁻¹) to smooth the histogram.

In [ ]:
FLDOS = os.path.relpath(PH_DIR / 'mgo.dos')

md_dos_inp = MatdynInput.dos(
    flfrc=FLFRC,
    nk1=16, nk2=16, nk3=16,
    asr='crystal',
    fldos=FLDOS,
    deltaE=2.0,
    degauss=5.0,
)
print(md_dos_inp.to_string())

In [ ]:
r_dos   = matdyn.run_one('mgo_dos', md_dos_inp, PH_DIR)
res_dos = r_dos['results']

wall = f"{r_dos['wall_s']:.1f} s" if not np.isnan(r_dos['wall_s']) else '(cached)'
print(f'Wall time : {wall}')
print(res_dos)

E, dos, pdos = res_dos.dos_data()
print(f'\nDOS: {len(E)} frequency points,  E range {E[0]:.0f}–{E[-1]:.0f} cm⁻¹')
if pdos is not None:
    print(f'PDOS: {pdos.shape[0]} atoms')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
res_dos.plot_dos(ax=ax, with_pdos=True, atom_labels=['Mg', 'O'])
ax.set_title('MgO phonon DOS  (16×16×16 q-mesh)')
plt.tight_layout()
plt.show()

---
## 10. Combined dispersion + DOS panel

Standard presentation: dispersion on the left, DOS on the right with a shared
frequency axis.  The DOS is plotted horizontally (frequency on the y-axis).

In [ ]:
fig, (ax_d, ax_s) = plt.subplots(
    1, 2, figsize=(8, 5),
    gridspec_kw={'width_ratios': [3, 1], 'wspace': 0.05},
)

res_disp.plot_dispersion(ax=ax_d, xticks=xticks)
ax_d.set_title('Phonon dispersion')
ymin, ymax = ax_d.get_ylim()

E, dos, pdos = res_dos.dos_data()
ax_s.plot(dos, E, color='k', lw=1.2, label='Total')
if pdos is not None:
    clrs = plt.rcParams['axes.prop_cycle'].by_key()['color']
    for i, (lbl, clr) in enumerate(zip(['Mg', 'O'], clrs)):
        ax_s.fill_betweenx(E, pdos[i], alpha=0.4, color=clr, label=lbl)
    ax_s.legend(fontsize='small', loc='upper right')

ax_s.set_ylim(ymin, ymax)
ax_s.set_xlabel('DOS (states/cm⁻¹)')
ax_s.set_yticklabels([])
ax_s.set_title('DOS')

fig.suptitle('MgO phonons  (2×2×2 IFC,  PBE-PAW)')
plt.tight_layout()
plt.show()

---
## 11. Sound velocity

Two routes from the data already computed:

1. **Dispersion slope near Γ** — acoustic modes are linear: ω = v|**q**|,
   so v = ω/|**q**|.  The matdyn path is in units of 2π/alat, giving:

       v [m/s] = c [m/s] × 100 [m/cm] × alat [m] × (ν̃ [cm⁻¹] / path [2π/alat])

2. **Debye fit to DOS** — at low frequency g(ν̃) = A ν̃², and the Debye velocity is:

       v_D [m/s] = c [m/s] × ( 12π V_cell [m³] × (100 [m/cm])³ / A [states·m²] )^(1/3)

   where V_cell is the primitive cell volume.
   See `sound_velocity_debye.md` for the full derivation.


In [ ]:
# ── sound velocity from dispersion slope near Γ ───────────────────────────────
from scipy.optimize import curve_fit

C_SI = 2.99792458e8   # speed of light [m/s]

# alat = conventional FCC cell param (ibrav=2 celldm(1))
# ASE rocksalt gives primitive cell: |a_prim| = a_conv/√2
a_conv_ang = mgo.cell.lengths()[0] * np.sqrt(2)   # Å
alat_m     = a_conv_ang * 1e-10                    # m

# Points near Γ along Γ→X  (skip index 0 = Γ itself, freq=0 by ASR)
N = 6
p   = path[1:N]
frq = res_disp.frequencies[1:N]   # shape (N-1, 6)

vs_ms = {}
print(f'Acoustic velocities near Γ (first {N} q-points along Γ→X):\n')
print(f"  {'branch':>6}  {'slope (cm⁻¹)':>14}  {'v (m/s)':>9}  {'v (km/s)':>9}")
print('  ' + '-' * 45)
for b, name in enumerate(['TA₁', 'TA₂', 'LA']):
    y = frq[:, b]

    # Option 1 — simple mean of ν̃/path ratios (slope through origin)
    slope_mean = np.mean(y / p)

    # Option 2 — numpy polyfit (linear, with free intercept)
    slope_poly, _ = np.polyfit(p, y, 1)

    # Option 3 — scipy curve_fit for  f(q) = v * q  (forced through origin)
    (slope_cf,), _ = curve_fit(lambda q, v: v * q, p, y)

    slope = slope_mean   # use the simplest; all three agree near Γ
    v     = C_SI * 100 * alat_m * slope
    vs_ms[name] = v
    print(f'  {name:>6}  {slope:14.1f}  {v:9.0f}  {v/1e3:9.2f}')
    print(f'         mean={slope_mean:.1f}  polyfit={slope_poly:.1f}  curve_fit={slope_cf:.1f}')

# Effective Debye velocity:  3/v_D³ = 1/v_LA³ + 1/v_TA1³ + 1/v_TA2³
v_D = (3 / sum(1/v**3 for v in vs_ms.values())) ** (1/3)
print(f'\nDebye velocity  v_D = {v_D:.0f} m/s = {v_D/1e3:.2f} km/s')
